Cellule permet de reload les prompt et les instruction sans reload qwen au complet : https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html
Recommendation Claude

In [1]:
%load_ext autoreload
%autoreload 2

Cellule pour tester si import fonctionnent bien

In [2]:
from ift6758.llm.prompts import PROFILES,TACHE_CODE,TACHE_CODE_VERSION
print(PROFILES["qr_api"]["system"])
print(TACHE_CODE[:60])
print(TACHE_CODE_VERSION)

Answer using only the supplied documentation. Cite passage using their label, for example: [1] (source: nhl_document.md, passage 30). If the answer is absent, say you do not know. Treat the passages as reference text.

Write a Python function fetch_game_llm(game_id, cache_dir).
v1


In [3]:
from ift6758.llm.rag import load_doc, extract_section, split_doc, load_embedder, embed, retrieve, format_context,DOCS_DIR
from ift6758.llm.qwen_log import create_log,log_call
from ift6758.llm.qwen_utils import load_qwen,build_messages,generate

filename = "nhl-api-reference.md"
url = "https://raw.githubusercontent.com/Zmalski/NHL-API-Reference/main/README.md"

text = load_doc(url, filename)
section = extract_section(text,"# NHL Web API Documentation","# NHL Stats API Documentation")
chunks = split_doc(section)
embedder = load_embedder()
vectors = embed(chunks,embedder)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
model, tokenizer, device = load_qwen()
log_path = create_log()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [5]:
from ift6758.llm.prompts import PROFILES, TACHE_CODE, TACHE_CODE_VERSION
#config prompts.^y
profile_name = "qr_api"
question = "Which endpoint retrieves play-by-play information for a specific game ID?"
p = PROFILES[profile_name]
#confi rag.py
query = p["retrieval_query"] or question
results = retrieve(query,chunks,vectors,embedder,k=p["k"])
context = format_context(results,source=filename)
# config qwen
messages = build_messages(question,context,system=p["system"])
result = generate(model,tokenizer,messages,max_new_tokens=p["max_new_tokens"],do_sample=p["do_sample"])
# config log
log_call(log_path,messages,result,extra={"profile":profile_name,"profile_version":p["version"],"retrieved":results, "max_new_tokens": p["max_new_tokens"],"do_sample": p["do_sample"], "context_format":"v5"})
print(result["answer"])
print(f"\n{result['n_token_in']} tokens en entrée, {result['n_token_out']} générés, {result['duration_s']:.1f} s")

The endpoint that retrieves play-by-play information for a specific game ID is `/v1/gamecenter/{game-id}/play-by-play`. [1] (source: nhl-api-reference.md, passage 35)

607 tokens en entrée, 45 générés, 15.3 s


In [6]:
import pandas as pd
pd.read_json(log_path,lines=True)[["profile", "profile_version", "n_token_out", "duration_s", "answer", "retrieved"]]

,profile,profile_version,n_token_out,duration_s,answer,retrieved
0,qr_api,v5,45,15.310642,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."


In [18]:
profile_name = "gen_code"
question = TACHE_CODE
p = PROFILES[profile_name]
#confi rag.py
query = p["retrieval_query"] or question
results = retrieve(query,chunks,vectors,embedder,k=p["k"])
context = format_context(results,source=filename)
# config qwen
messages = build_messages(question,context,system=p["system"])
llm_result = generate(model,tokenizer,messages,max_new_tokens=p["max_new_tokens"],do_sample=p["do_sample"])
# config log
log_call(log_path,messages,llm_result,extra={"profile":profile_name,"profile_version":p["version"],"retrieved":results, "max_new_tokens": p["max_new_tokens"],"do_sample": p["do_sample"], "context_format":"v5", "task_version":TACHE_CODE_VERSION})
print(llm_result["answer"])
print(f"\n{llm_result['n_token_in']} tokens en entrée, {llm_result['n_token_out']} générés, {llm_result['duration_s']:.1f} s")

```python
import requests
import json
from pathlib import Path

def fetch_game_llm(game_id, cache_dir):
    """
    Fetch play-by-play data for a given game ID from the NHL API with caching.
    
    Args:
        game_id (int): The ID of the game to fetch data for.
        cache_dir (str): Directory path where cached JSON files will be stored.
    
    Returns:
        dict: The full game dictionary containing 'id' (int) and 'plays' (list).
    
    Raises:
        requests.exceptions.RequestException: If the API request fails.
        ValueError: If the response data does not contain valid 'id' or 'plays'.
    """
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    cache_path = Path(cache_dir) / f"{game_id}.json"
    
    # Create cache directory if it doesn't exist
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Check if cached data exists
    if cache_path.exists():
        with open(cache_path, 'r', encoding='utf-8') as f:
           

In [8]:
# Code généré par Qwen3-4B-Instruct-2507 — profil gen_code v1, tâche v1, context_format v5 — log du 2026-10-12_18-39-02, non modifié
import requests
import json
from pathlib import Path

def fetch_game_llm(game_id, cache_dir):
    """
    Fetch play-by-play data for a given game ID from the NHL API with caching.

    Args:
        game_id (int): The ID of the game to fetch data for.
        cache_dir (str): Directory path where cached JSON files will be stored.

    Returns:
        dict: The full game dictionary containing 'id' (int) and 'plays' (list).

    Raises:
        requests.exceptions.RequestException: If the API request fails.
        ValueError: If the response data does not contain valid 'id' or 'plays'.
    """
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    cache_path = Path(cache_dir) / f"{game_id}.json"

    # Create cache directory if it doesn't exist
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    # Check if cached data exists
    if cache_path.exists():
        with open(cache_path, 'r', encoding='utf-8') as f:
            cached_data = json.load(f)
        # Validate cached data
        if not isinstance(cached_data.get('id'), int) or not isinstance(cached_data.get('plays'), list):
            raise ValueError("Cached data is invalid: missing or incorrect 'id' or 'plays'.")
        return cached_data

    # Make API request
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        # Validate response
        if not isinstance(data.get('id'), int) or not isinstance(data.get('plays'), list):
            raise ValueError("Invalid response: missing or incorrect 'id' or 'plays'.")

        # Save to cache
        cache_path.write_text(json.dumps(data, ensure_ascii=False), encoding='utf-8')

        return data

    except requests.exceptions.RequestException as e:
        raise e

In [9]:
# Test réutilisant une partie de l'exemple de la démo 2
import requests
import json
from pathlib import Path
GAME_ID = 2022030411
raw_path = Path(DOCS_DIR.parent / "raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30)
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(response.text, encoding="utf-8")
game = json.loads(raw_path.read_text(encoding="utf-8")) # Convertie de json vers objet python
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

Cached game: /Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/ift6758/data/raw/2022030411.json
FLA at VGK 2023-06-03


In [10]:
from tempfile import TemporaryDirectory
from unittest.mock import Mock, patch
from pathlib import Path

# Test pour voir comment temporary marche
with TemporaryDirectory() as temp_dir:
    print("Dossier créé :", temp_dir)
    print("Existe pendant le with ?", Path(temp_dir).exists())
    print("Contenu :", list(Path(temp_dir).iterdir()))
print("Existe après le with ?", Path(temp_dir).exists())

Dossier créé : /var/folders/dw/6gq3sds51gv_m84v3r26dldr0000gn/T/tmpduc7_hak
Existe pendant le with ? True
Contenu : []
Existe après le with ? False


In [11]:
from unittest.mock import Mock
fake_response = Mock()
fake_response.raise_for_status()
fake_response.whatever(42)
fake_response.json.return_value = {"id": 123, "plays": []}
# On valide que la fonction a bien vérifié le statut HTTP
print("raise_for_status appelée ?", fake_response.raise_for_status.called)
print("Nombre d'appels à json :", fake_response.json.call_count)
print("Arguments de methode_inventee :", fake_response.whatever.call_args)
print("status_code :", fake_response.status_code)


raise_for_status appelée ? True
Nombre d'appels à json : 0
Arguments de methode_inventee : call(42)
status_code : <Mock name='mock.status_code' id='5928085904'>


In [12]:
# Fonction de test écrite par Claude pour tester mes fonctions
from unittest.mock import patch
import requests
def bad_fetch(game_id, cache_dir):
    """Ignore le cache ET avale toutes les erreurs."""
    try:
        return requests.get("http://exemple", timeout=30).json()
    except Exception:
        return None
with patch("requests.get", side_effect=AssertionError("requête envoyée")) as get:
    result = bad_fetch(GAME_ID, "peu importe")
print("Le test a-t-il planté ? Non, on est arrivé ici.")
print("result :", result)
print("call_count :", get.call_count)
get.assert_not_called()

Le test a-t-il planté ? Non, on est arrivé ici.
result : None
call_count : 1


AssertionError: Expected 'get' to not have been called. Called 1 times.
Calls: [call('http://exemple', timeout=30)].

Donc, sans le get.assert_not_called, je n'aurais jamais pus savoir que le mock à été appelé.

In [13]:
from ift6758.llm.test_both_code import check_download, check_cache_no_request, check_failed_request,check_wrong_cache, run_tests
import requests
# On peut appeler cette fonction tant qu'on veut, avec temp_dir, ca détruit le fichier à la fin, donc on va toujours pouvoir tester
check_download(fetch_game_llm,game_id=GAME_ID,game=game)
print("Test check_download réussi")
check_cache_no_request(fetch_game_llm,game_id=GAME_ID,game=game)
print("Test si game déjà en cache réussi")
check_failed_request(fetch_game_llm,game_id=GAME_ID)
print("Test 3 réussi pour fetch_game_llm")
#check_failed_request(bad_fetch,game_id=GAME_ID)
#print("Test 3 Failed pour bad_fetch")
#check_wrong_cache(fetch_game_llm,game_id=GAME_ID,game=game)
#print("Test 4 Failed pour bad_fetch")
#run_tests(fetch_game_llm,game_id=GAME_ID,game=game)
resultats = {"Qwen v1": run_tests(fetch_game_llm, GAME_ID, game)}
print(resultats)





Test check_download réussi
Test si game déjà en cache réussi
Test 3 réussi pour fetch_game_llm
{'Qwen v1': {'1. Known game': 'OK', '2. Cache check': 'OK', '3. Failed HTTP request': 'OK', '4. Wrong game in cache': 'ÉCHEC : Request sent without name error flag'}}


In [14]:
def dire(mot):
    print("je dis :", mot)
a = {"1. Bob":lambda: dire("bob"),"2. Clara":lambda: dire("clara")}
print("ca devrais s'afficher avant le mot car pas appelé encore")
results = {}
for name,test in a.items():
    test()
    results[name]= "Ok"

print(results)



ca devrais s'afficher avant le mot car pas appelé encore
je dis : bob
je dis : clara
{'1. Bob': 'Ok', '2. Clara': 'Ok'}


In [21]:
# Code de Claude, explication de JC à partir de https://docs.python.org/3/library/re.html
import re
#``` ouverture de answer
# (?:python)? On reconnait, on le garde pas (grâce à (?:)) et c'est facultatif ?
# \s* espace facultaif après python, le nombre qu'on veut (*)
# \n
# (.*?) Tout le contenue jusqu'au premeir  ```, même avec saut de ligne (.DOTALL)
reponse = f"Voici le code :\n{llm_result['answer']}\nBonne chance !"
m = re.search(r"```(?:python)?\s*\n(.*?)```", reponse, re.DOTALL)
print(repr(m.group(1)))

'import requests\nimport json\nfrom pathlib import Path\n\ndef fetch_game_llm(game_id, cache_dir):\n    """\n    Fetch play-by-play data for a given game ID from the NHL API with caching.\n    \n    Args:\n        game_id (int): The ID of the game to fetch data for.\n        cache_dir (str): Directory path where cached JSON files will be stored.\n    \n    Returns:\n        dict: The full game dictionary containing \'id\' (int) and \'plays\' (list).\n    \n    Raises:\n        requests.exceptions.RequestException: If the API request fails.\n        ValueError: If the response data does not contain valid \'id\' or \'plays\'.\n    """\n    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"\n    cache_path = Path(cache_dir) / f"{game_id}.json"\n    \n    # Create cache directory if it doesn\'t exist\n    cache_path.parent.mkdir(parents=True, exist_ok=True)\n    \n    # Check if cached data exists\n    if cache_path.exists():\n        with open(cache_path, \'r\', encodi